# Week 4 — Streaming video compositing

This lab replaces the old per-pixel loops and TIFF directories with a streaming, vectorised pipeline. The supplied videos have different resolutions and frame rates, so the pipeline must inspect and synchronise them rather than rely on hard-coded values.

> **Reference code and academic integrity**
>
> Some completed code cells in this notebook are supplied as runnable reference examples. They demonstrate the method and keep the notebook executable from start to finish; they are not model answers and are not authorised for reuse in assessed work. Your submitted code must be your own, and you must be able to explain it. Copying, lightly modifying, translating, or closely reproducing this reference code in a submission is plagiarism. **Any submission found to plagiarise the supplied reference code will receive zero marks** and may also be referred under the University Academic Integrity Policy.


## Learning outcomes

By the end of the lab you should be able to:

- explain why HSV is useful for chroma keying;
- refine a binary mask with morphology and feather its boundary;
- express alpha compositing using NumPy broadcasting;
- process video one frame at a time while respecting source FPS; and
- validate an encoded result rather than trusting a file extension.

In [ ]:
from pathlib import Path
import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Video, display

candidates = [
    Path.cwd(),
    Path.cwd() / "COMP3419-W04-Lab-Files",
]
WEEK_DIR = next((path for path in candidates if (path / "video_compositing.py").is_file()), None)
if WEEK_DIR is None:
    raise FileNotFoundError("Run this notebook from its folder or the repository root")
sys.path.insert(0, str(WEEK_DIR.resolve()))

from video_compositing import (
    BACKGROUND_VIDEO,
    FOREGROUND_VIDEO,
    BlueScreenKeyer,
    alpha_composite,
    blue_screen_mask,
    composite_videos,
    foreground_alpha,
    inspect_video,
    plot_samples,
    resize_to_cover,
    validate_video,
)

## 1. Inspect the sources

The old notebook accidentally implied `monkey.mov`; the supplied foreground is `W4LabData/monkey.avi`. Notice that the source FPS values differ.

In [ ]:
foreground_info = inspect_video(FOREGROUND_VIDEO)
background_info = inspect_video(BACKGROUND_VIDEO)
foreground_info, background_info

In [ ]:
plot_samples(FOREGROUND_VIDEO, count=3)
plot_samples(BACKGROUND_VIDEO, count=3)

## 2. Build and inspect a key

For foreground colour $F$, replacement background $B$ and foreground opacity $\alpha$, compositing is

$$C = \alpha F + (1-\alpha)B.$$

HSV separates hue from saturation/value, making a blue interval easier to interpret than three independent BGR thresholds. Opening removes isolated mask noise; closing fills small holes; Gaussian feathering softens the boundary.

**TODO checkpoint:** adjust the HSV bounds and radii below. Record one setting that removes more blue but damages the subject, then justify your final trade-off.

In [ ]:
# Safe starting point; edit these values for the checkpoint.
keyer = BlueScreenKeyer(
    lower_hsv=(90, 45, 20),
    upper_hsv=(140, 255, 255),
    morphology_radius=2,
    feather_radius=2,
)
keyer

In [ ]:
fg_capture = cv2.VideoCapture(str(FOREGROUND_VIDEO))
bg_capture = cv2.VideoCapture(str(BACKGROUND_VIDEO))
fg_capture.set(cv2.CAP_PROP_POS_FRAMES, 100)
ok_fg, foreground_bgr = fg_capture.read()
ok_bg, background_bgr = bg_capture.read()
fg_capture.release()
bg_capture.release()
if not (ok_fg and ok_bg):
    raise RuntimeError("Could not decode preview frames")

background_bgr = resize_to_cover(
    background_bgr, foreground_bgr.shape[1], foreground_bgr.shape[0]
)
screen_mask = blue_screen_mask(foreground_bgr, keyer)
alpha = foreground_alpha(screen_mask, keyer.feather_radius)
composite_bgr = alpha_composite(foreground_bgr, background_bgr, alpha)

figure, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(cv2.cvtColor(foreground_bgr, cv2.COLOR_BGR2RGB)); axes[0].set_title("foreground")
axes[1].imshow(screen_mask, cmap="gray", vmin=0, vmax=255); axes[1].set_title("screen mask")
axes[2].imshow(alpha, cmap="gray", vmin=0, vmax=1); axes[2].set_title("foreground alpha")
axes[3].imshow(cv2.cvtColor(composite_bgr, cv2.COLOR_BGR2RGB)); axes[3].set_title("composite")
for axis in axes: axis.axis("off")
figure.tight_layout()

## 3. Stream the output

`composite_videos` reads, keys, composites and writes one frame per iteration. It maps foreground frame time to background frame time, so a 10 FPS background is not unintentionally played at 25 FPS. Start with a short preview; use `None` for the complete 752-frame result.

In [ ]:
OUTPUT_DIR = WEEK_DIR / "outputs"
OUTPUT_VIDEO = OUTPUT_DIR / "w4_composite.mp4"
MAX_FRAMES = 75  # TODO: set to None after checking the preview

report = composite_videos(
    FOREGROUND_VIDEO,
    BACKGROUND_VIDEO,
    OUTPUT_VIDEO,
    keyer=keyer,
    max_frames=MAX_FRAMES,
)
report

In [ ]:
display(Video(str(OUTPUT_VIDEO), embed=True, width=720))
plot_samples(OUTPUT_VIDEO, count=4)

## 4. Validate and explain

Encoding can fail even when a filename exists. Re-open the output and verify its decoded frame count, size and FPS.

In [ ]:
validated = validate_video(
    OUTPUT_VIDEO,
    expected_frames=report.frames_written,
    expected_fps=report.fps,
    expected_size=(report.width, report.height),
)
assert 0.0 < report.mean_screen_fraction < 1.0
validated

## Checkpoint submission

Include representative foreground/mask/composite frames, the validated metadata, your chosen parameters and answers to these questions:

1. Why is broadcasting over a full mask preferable to nested Python pixel loops?
2. What visual errors do opening, closing and feathering each address?
3. What would happen if background frame $i$ were paired directly with foreground frame $i$ despite their different FPS values?
4. Identify one failure case in your result and propose a technically plausible improvement (for example colour spill suppression or a learned matte).